<a href="https://colab.research.google.com/github/Fdiannz/BT4222-Project-Group-4--MIND/blob/main/step4_advanced_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# step4_advanced_model: NRMS (Neural News Recommendation with Multi-Head Self-Attention)

**Owner:** P5
**Purpose:** a content-based neural recommender that scores articles from their **title words** and the user's **click history**. Because it judges articles by their content rather than their click counts, it can rank newly published (cold) articles.
**Inputs:** `data/processed/` from step 1: `news_metadata`, `fit_*` (9–13 Nov), `tune_*` (14 Nov), `eval_*` (15 Nov).
**Outputs (shared):**
- `data/scores/step4_nrms_{tune,eval,train_sample}.parquet`: one row per displayed candidate, columns `impression_id, news_id, score` (raw dot product, higher = more relevant). Used by step 5 (MMR) and step 6 (diagnostics).
- `data/embeddings/step4_nrms_news_vectors.npy` + `step4_nrms_news_ids.parquet`: the model's 256-d article vectors and their article IDs (same row order).
- `outputs/step4_nrms/`: checkpoints, training logs, result tables.

**Runtime:** Colab **GPU** (*Runtime → Change runtime type → T4 GPU*). Run top to bottom and keep the outputs. With `RESUME = True`, finished runs are loaded from their checkpoints instead of being retrained.

### How NRMS works (Wu et al., 2019)
1. **News encoder:** title words → word vectors → multi-head self-attention between the words → additive attention pools the words into one article vector.
2. **User encoder:** each clicked history article goes through the same news encoder → multi-head self-attention between the history articles → additive attention pools them into one user vector.
3. **Click score** = user vector · candidate article vector.

### Where each part comes from in the course

| NRMS component | Lecture / notebook |
|---|---|
| Pre-trained word embeddings (word2vec-google-news-300) | Topic 6 Embedding |
| Tokenise, pad, embedding lookup | Topic 7 (Sentiment Analysis LSTM example) |
| Self-attention, multi-head attention | Topic 7 Part B (Transformer) |
| Attention pooling, inspecting attention weights | Sequential Recommendation LSTM with Attention notebook |
| 1 click + 4 within-impression negatives, softmax cross-entropy on raw scores | Topics 6/7 negative sampling and BPR; ML Optimization notebook |
| Per-impression AUC / MRR / nDCG on full slates; separate tune and test sets | Topic 4 Evaluation (restricted candidate set bias) |

### Standard NRMS setup vs this notebook

| Setting | Standard NRMS (paper / MIND benchmark) | Here | Why |
|---|---|---|---|
| Input text | Title words, up to 30 | Same | |
| Word vectors | 300-d GloVe, fine-tuned | 300-d **word2vec-google-news-300**, fine-tuned | Lecture embedding (Topic 6); news-domain corpus |
| News self-attention | 16 heads × 16 dims = 256 | 16 heads, 256 dims after a linear 300→256 projection | PyTorch `MultiheadAttention` keeps its input size, so the projection gives the same 256-d output |
| Additive attention | Query size 200 | Same | |
| Dropout | 0.2 | Same | |
| History length | 50 most recent clicks | Same | |
| Click predictor | Dot product | Same | |
| Training | 1 click + K=4 negatives from the same impression, softmax cross-entropy, Adam | Same (lr 1e-4, batch 64) | |
| Users with no history | Not specified (padding) | Learned "empty user" vector | About 2–3% of impressions |
| Training data | Full training log | Random subset of 1.5M clicks from 9–13 Nov | Colab GPU time; stated in the report |
| Evaluation | AUC, MRR, nDCG@5/10 per impression | Same, via shared `utils/metrics.py`, on full slates | |

## 0. Setup

In [1]:
# --- Setup: run this first, every session ---
from google.colab import drive
drive.mount('/content/drive')

# Fresh copy of the team repo (shared config.py and utils/) on every run
!rm -rf /content/BT4222-Project-Group-4--MIND && git clone -q https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git /content/BT4222-Project-Group-4--MIND
import sys; sys.path.insert(0, '/content/BT4222-Project-Group-4--MIND')
!pip -q install gensim

import config
print("Data folder:", config.PROCESSED_DIR)

Mounted at /content/drive
Data folder: /content/drive/MyDrive/BT4222_Project/data/processed


## 1. Settings

All experiment settings live in `CFG`. `COLS` maps the standard names used in this notebook to the column names in step 1's files.

`RUN` names the checkpoint files. With `RESUME = True`, a run that already finished is loaded from Drive instead of being retrained. **Change `RUN` whenever you change `CFG`**, so an old checkpoint is never resumed with new settings.

In [2]:
import os, re, json, time, math, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from utils.metrics import evaluate, evaluate_by_segment, attach_scores, label_impression_segment

# ---- folders (all from the shared config.py) ----
PROC = config.PROCESSED_DIR
DATA_DIR = getattr(config, "DATA_DIR", os.path.dirname(PROC))
OUTPUT_DIR = getattr(config, "OUTPUT_DIR", os.path.join(os.path.dirname(DATA_DIR), "outputs"))
SCORE_DIR = getattr(config, "SCORES_DIR", os.path.join(DATA_DIR, "scores"))   # shared score files
EMB_DIR = os.path.join(DATA_DIR, "embeddings")                                  # shared article vectors
RUN_DIR = os.path.join(OUTPUT_DIR, "step4_nrms")                                # checkpoints, logs, tables
for d in (SCORE_DIR, EMB_DIR, RUN_DIR):
    os.makedirs(d, exist_ok=True)

FILES = {
    "news": "news_metadata.parquet",
    "fit_imp": "fit_impressions.parquet",   "fit_cand": "fit_candidates_sampled.parquet",
    "tune_imp": "tune_impressions.parquet", "tune_cand": "tune_candidates.parquet",
    "eval_imp": "eval_impressions.parquet", "eval_cand": "eval_candidates.parquet",
}

# standard name -> column name in step 1's files
COLS = {
    "impression_id": "impression_id",
    "news_id": "news_id",
    "label": "label",
    "is_cold": "cold_article",                  # 1 if the article had < config.COLD_THRESHOLD exposures on earlier days
    "prior_count": "prior_exposure_count",      # exposures on earlier days (used only for checks)
    "time": "time",
    "history": "history",
    "impressions": "impressions",               # full slate string 'N1-0 N2-1 ...' in *_impressions
    "title": "title",
    "abstract": "abstract",
    "category": "category",
}

CFG = dict(
    text_fields=("title",), field_lens=(30,),  # words kept per text field
    max_hist=50,              # most recent history clicks per user
    neg_k=4,                  # negatives per click, all from the same impression
    d_model=256, n_heads=16, query_dim=200, dropout=0.2,
    lr=1e-4, batch_size=64, epochs=6,
    n_train_clicks=1_500_000, # random subset of fit clicks (compute budget); stated in the report
    n_tune_imps=20_000,       # fixed tune subset scored after each epoch to pick the best epoch
    n_train_slate_imps=20_000,# training impressions re-scored on full slates (train vs test comparison)
    w2v_limit=500_000,        # most frequent word2vec words loaded (saves RAM)
    seed=getattr(config, "SEED", 42),
)
RUN = "nrms_main_v2"        # checkpoint name of the main model (1.5M clicks, 6 epochs)
USE_W2V = True              # False = random word vectors (quick dry run only)
RESUME = True               # load a finished run / continue an interrupted one
REFIT_FINAL = False         # optional: retrain on 9-14 Nov after model selection (section 11b)
RUN_ABLATIONS = True        # section 12; finished ablations reload from their checkpoints

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"     # mixed precision on GPU: faster, same results up to rounding

def seed_everything(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])
print("Device:", DEVICE, "| run:", RUN)

Device: cuda | run: nrms_main_v2


## 2. Load step 1 data and run bias checks

Only the needed columns are loaded, and they are renamed to the standard names in `COLS`.

The checks below guard against the most common sources of bias before any training:
1. **No time overlap:** every fit impression is before every tune impression, which is before every test impression.
2. **Cold labels use earlier days only:** within one day, an article's prior exposure count must not change. A count that grows during the day would mean exposures from the same day were counted, so new articles would turn "warm" within minutes.
3. **Candidates have article text:** every scored article exists in `news_metadata`.

In [3]:
def load(name, want):
    # Read only the wanted columns that exist in the file, renamed to standard names.
    path = os.path.join(PROC, FILES[name])
    have = pq.read_schema(path).names
    cols = [COLS[c] for c in want if COLS[c] in have]
    df = pd.read_parquet(path, columns=cols).rename(columns={COLS[c]: c for c in want})
    print(f"{name:10s} {len(df):>12,} rows | columns used: {list(df.columns)}")
    return df, have

NEED = {
    "news": ["news_id", "title"],
    "fit_imp": ["impression_id", "history"], "fit_cand": ["impression_id", "news_id", "label"],
    "tune_imp": ["impression_id", "history"], "tune_cand": ["impression_id", "news_id", "label", "is_cold"],
    "eval_imp": ["impression_id", "history"], "eval_cand": ["impression_id", "news_id", "label", "is_cold"],
}
OPTIONAL = {"news": ["abstract", "category"],
            "fit_cand": ["time", "prior_count"],
            "tune_cand": ["category", "time", "prior_count"],
            "eval_cand": ["category", "time", "prior_count"]}

data, schemas = {}, {}
for name in FILES:
    data[name], schemas[name] = load(name, NEED[name] + OPTIONAL.get(name, []))

missing = {n: [c for c in NEED[n] if c not in data[n].columns] for n in NEED}
missing = {n: c for n, c in missing.items() if c}
assert not missing, f"Missing columns {missing}: update COLS in section 1."

news = data["news"].drop_duplicates("news_id").reset_index(drop=True)
fit_cand, fit_imp = data["fit_cand"], data["fit_imp"]
tune_cand, tune_imp = data["tune_cand"], data["tune_imp"]
eval_cand, eval_imp = data["eval_cand"], data["eval_imp"]

# ---- check 1: no time overlap between fit, tune and test ----
if all("time" in d for d in (fit_cand, tune_cand, eval_cand)):
    spans = {n: (d.time.min(), d.time.max()) for n, d in [("fit", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]}
    for n, (a, b) in spans.items():
        print(f"{n:5s} {a} -> {b}")
    assert spans["fit"][1] < spans["tune"][0] and spans["tune"][1] < spans["eval"][0], "splits overlap in time"
    print("check 1 passed: fit < tune < test in time")

# ---- check 2: cold labels are frozen within each day (earlier-day exposures only) ----
for n, d in [("tune", tune_cand), ("eval", eval_cand)]:
    if "prior_count" in d and "time" in d:
        n_values = d.groupby([d["news_id"], d["time"].dt.date])["prior_count"].nunique()
        frozen = (n_values == 1).mean()
        status = "passed" if frozen == 1 else "WARNING: counts change within a day (same-day exposures counted?)"
        print(f"check 2 {n}: {frozen:.1%} of article-days have one prior count -> {status}")

# ---- check 3: every candidate has article text ----
known = set(news["news_id"])
for n, d in [("fit", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]:
    miss = (~d["news_id"].isin(known)).sum()
    assert miss == 0, f"{miss:,} {n} candidates are missing from news_metadata"
print("check 3 passed: all candidates have article text")

for n, d in [("fit (sampled)", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]:
    extra = f", cold share {d.is_cold.mean():.3f}" if "is_cold" in d else ""
    print(f"{n:14s}: {d.impression_id.nunique():>9,} impressions, click rate {d.label.mean():.3f}{extra}")

news            104,151 rows | columns used: ['news_id', 'title', 'abstract', 'category']
fit_imp       1,801,231 rows | columns used: ['impression_id', 'history']
fit_cand     12,894,210 rows | columns used: ['impression_id', 'news_id', 'label', 'time', 'prior_count']
tune_imp        431,517 rows | columns used: ['impression_id', 'history']
tune_cand    17,400,106 rows | columns used: ['impression_id', 'news_id', 'label', 'is_cold', 'category', 'time', 'prior_count']
eval_imp        376,471 rows | columns used: ['impression_id', 'history']
eval_cand    14,085,557 rows | columns used: ['impression_id', 'news_id', 'label', 'is_cold', 'category', 'time', 'prior_count']
fit   2019-11-09 00:00:00 -> 2019-11-13 23:59:57
tune  2019-11-14 00:00:00 -> 2019-11-14 23:59:59
eval  2019-11-15 00:00:00 -> 2019-11-15 23:59:43
check 1 passed: fit < tune < test in time
check 2 tune: 100.0% of article-days have one prior count -> passed
check 2 eval: 100.0% of article-days have one prior count -> passed

## 3. Tokenise article text

Each article becomes a fixed-length array of word IDs (lower-cased, padded with 0), like the tokenise-and-pad step in the Topic 7 LSTM example. Row 0 is a padding article used to fill short histories.

The vocabulary uses article **text only** (titles and abstracts of all articles, no clicks or labels). An article's text is known as soon as it is published, so using test-day titles here is not leakage. Words seen only once map to `<unk>`.

Do not change the tokenizer or vocabulary rules after training: the saved checkpoints depend on the exact vocabulary.

In [4]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower()) if isinstance(text, str) else []

# news_id -> row in the token matrix (row 0 = padding article)
NEWS_ROW = pd.Series(np.arange(1, len(news) + 1), index=news["news_id"].to_numpy())

def build_vocab(fields):
    counts = {}
    for f in fields:
        for t in news[f].fillna("") if f in news else []:
            for w in tokenize(t):
                counts[w] = counts.get(w, 0) + 1
    words = sorted(w for w, c in counts.items() if c >= 2)        # rare words -> <unk>
    vocab = {"<pad>": 0, "<unk>": 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab

def build_tokens(fields, lens, vocab):
    # [n_articles + 1, sum(lens)] word IDs; each field gets its own fixed slot (e.g. 30 title words)
    tok = np.zeros((len(news) + 1, sum(lens)), dtype=np.int64)
    start = 0
    for f, L in zip(fields, lens):
        col = news[f].fillna("") if f in news else pd.Series([""] * len(news))
        for i, text in enumerate(col):
            ids = [vocab.get(w, 1) for w in tokenize(text)][:L]
            tok[i + 1, start:start + len(ids)] = ids
        start += L
    return tok

VOCAB = build_vocab(("title", "abstract") if "abstract" in news else ("title",))
TOKENS = build_tokens(CFG["text_fields"], CFG["field_lens"], VOCAB)
title_len = (TOKENS[1:, :CFG["field_lens"][0]] > 0).sum(1)
print(f"Vocabulary: {len(VOCAB):,} words | articles: {len(news):,} | "
      f"title words kept: median {np.median(title_len):.0f}, empty titles {np.mean(title_len == 0):.2%}")
print("Example:", news.loc[0, "title"], "->", TOKENS[1, :12])

Vocabulary: 53,058 words | articles: 104,151 | title words kept: median 11, empty titles 0.00%
Example: The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By -> [47395  7206 38098 16142 37334  9461  3274 37334 35954 46352  8127     0]


## 4. Word vectors: word2vec-google-news-300 (Topic 6)

Each vocabulary word starts from its pre-trained word2vec vector, which was trained on Google News, the same domain as MIND. Words missing from word2vec (including stopwords such as *a* and *of*) start from small random vectors. All vectors are fine-tuned during training.

The download (about 1.6 GB) happens once; the resulting matrix is cached on Drive.

In [5]:
EMB_DIM = 300
def build_word_vectors(vocab):
    cache = os.path.join(RUN_DIR, f"word2vec_init_{len(vocab)}.npy")
    if os.path.exists(cache):
        print("Loaded cached word vectors:", cache)
        return np.load(cache)
    rng = np.random.default_rng(CFG["seed"])
    emb = rng.normal(0, 0.1, (len(vocab), EMB_DIM)).astype(np.float32)
    emb[0] = 0.0                                          # padding word
    if not USE_W2V:
        print("USE_W2V=False: random word vectors (dry run only)")
        return emb
    import gensim.downloader as api
    from gensim.models import KeyedVectors
    path = api.load("word2vec-google-news-300", return_path=True)
    kv = KeyedVectors.load_word2vec_format(path, binary=True, limit=CFG["w2v_limit"])
    hits = 0
    for w, i in vocab.items():
        if i < 2:
            continue
        for form in (w, w.capitalize(), w.upper()):        # word2vec is case-sensitive
            if form in kv.key_to_index:
                emb[i] = kv[form]; hits += 1
                break
    print(f"word2vec coverage: {hits:,}/{len(vocab) - 2:,} words ({hits / (len(vocab) - 2):.1%})")
    np.save(cache, emb)
    del kv
    return emb

WORD_EMB = build_word_vectors(VOCAB)

Loaded cached word vectors: /content/drive/MyDrive/BT4222_Project/outputs/step4_nrms/word2vec_init_53058.npy


## 5. User histories

Each impression's click history becomes up to 50 article rows (most recent kept), padded with 0. In MIND, the history lists clicks made **before** the impression, so it never contains the impression's own outcome.

In [6]:
def parse_history(h):
    if h is None or (isinstance(h, float) and np.isnan(h)):
        return []
    return h.split() if isinstance(h, str) else list(h)

def build_history(imp_df, impression_ids=None):
    # Returns (H, index): H[n_imp, max_hist] holds article rows; index maps impression_id -> row of H.
    df = imp_df if impression_ids is None else imp_df[imp_df.impression_id.isin(impression_ids)]
    df = df.drop_duplicates("impression_id")
    H = np.zeros((len(df), CFG["max_hist"]), dtype=np.int64)
    for i, h in enumerate(df["history"].to_numpy()):
        ids = parse_history(h)[-CFG["max_hist"]:]           # keep the most recent clicks
        rows = NEWS_ROW.reindex(ids).fillna(0).to_numpy(dtype=np.int64) if ids else []
        H[i, :len(rows)] = rows
    return H, pd.Series(np.arange(len(df)), index=df["impression_id"].to_numpy())

tune_H, tune_idx = build_history(tune_imp)
eval_H, eval_idx = build_history(eval_imp)
for name, H in [("tune", tune_H), ("eval", eval_H)]:
    n = (H > 0).sum(1)
    print(f"{name}: {len(H):,} impressions | history length median {np.median(n):.0f}, empty {np.mean(n == 0):.2%}")

tune: 431,517 impressions | history length median 20, empty 2.64%
eval: 376,471 impressions | history length median 19, empty 2.99%


## 6. Training samples: 1 click + 4 negatives from the same impression

Each click in `fit_candidates_sampled` becomes one sample with 4 unclicked articles **from the same impression**, so every negative was actually shown to the user (interim Section 3.3). When an impression has fewer than 4 unclicked articles, some are reused; the share is printed.

The click is always placed first (target index 0). This creates no position bias: the model scores each candidate separately and never sees its position.

In [7]:
def build_train_samples(cand, n_clicks, k, seed):
    # Returns (impression_id per sample, C[n, 1 + k] article rows with the click in column 0).
    rng = np.random.default_rng(seed)
    cand = cand.assign(row=NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0).to_numpy(dtype=np.int64))
    pos = cand[cand.label == 1]
    if n_clicks and n_clicks < len(pos):
        pos = pos.sample(n=n_clicks, random_state=seed)                            # random click subset
    neg = cand[(cand.label == 0) & cand.impression_id.isin(pos.impression_id)]
    neg = neg.assign(r=rng.random(len(neg))).sort_values(["impression_id", "r"])   # shuffle within impression
    imp_ids, start, count = np.unique(neg.impression_id.to_numpy(), return_index=True, return_counts=True)
    where = pd.Series(np.arange(len(imp_ids)), index=imp_ids)
    pos = pos[pos.impression_id.isin(imp_ids)]                                     # need at least one negative
    g = where.reindex(pos.impression_id.to_numpy()).to_numpy()
    j = pos.groupby("impression_id").cumcount().to_numpy()                         # j-th click of its impression
    offs = (j[:, None] * k + np.arange(k)[None, :]) % count[g][:, None]            # different negatives per click
    neg_rows = neg["row"].to_numpy()[start[g][:, None] + offs]
    C = np.concatenate([pos["row"].to_numpy()[:, None], neg_rows], axis=1)
    return pos["impression_id"].to_numpy(), C

train_imp_ids, train_C = build_train_samples(fit_cand, CFG["n_train_clicks"], CFG["neg_k"], CFG["seed"])
fit_H, fit_idx = build_history(fit_imp, impression_ids=np.unique(train_imp_ids))
train_H_rows = fit_idx.reindex(train_imp_ids).to_numpy()
assert not np.isnan(train_H_rows.astype(float)).any(), "some training impressions have no history row"
reused = np.mean([len(set(r[1:])) < CFG["neg_k"] for r in train_C[:200_000]])
print(f"Training samples: {len(train_C):,} clicks x {train_C.shape[1]} candidates from "
      f"{len(np.unique(train_imp_ids)):,} impressions | samples with a reused negative: {reused:.1%}")

Training samples: 1,500,000 clicks x 5 candidates from 1,146,261 impressions | samples with a reused negative: 10.0%


## 7. Model

- `AdditiveAttention`: scores each item with a small learned query vector and returns the weighted average and the weights (so attention can be inspected in section 13).
- `NewsEncoder`: word vectors → dropout → 300→256 projection → 16-head self-attention over the words → dropout → additive attention over the words.
- `UserEncoder`: 16-head self-attention over the history article vectors → dropout → additive attention. Users with no history get a learned "empty user" vector.
- `NRMS.forward`: encodes each distinct article in the batch once, then returns **raw** dot-product scores. `F.cross_entropy` applies the softmax itself (ML Optimization notebook).

Layer names must stay as they are, because the saved checkpoints refer to them.

In [8]:
class AdditiveAttention(nn.Module):
    def __init__(self, dim, query_dim):
        super().__init__()
        self.proj = nn.Linear(dim, query_dim)
        self.query = nn.Linear(query_dim, 1, bias=False)

    def forward(self, x, mask):                       # x [N, L, d], mask [N, L] True = real item
        a = self.query(torch.tanh(self.proj(x))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)                # padding gets ~0 weight; -1e4 is safe in float16
        w = torch.softmax(a, dim=-1)
        return (w.unsqueeze(-1) * x).sum(1), w


def _safe_mask(mask):
    # Attention cannot run over a row with no real items, so unmask the first slot of empty rows.
    empty = ~mask.any(1)
    mask = mask.clone()
    mask[empty, 0] = True
    return mask, empty


class NewsEncoder(nn.Module):
    def __init__(self, word_emb, d_model, n_heads, query_dim, dropout):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.as_tensor(word_emb), freeze=False, padding_idx=0)
        self.proj = nn.Linear(word_emb.shape[1], d_model)
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.att = AdditiveAttention(d_model, query_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, tokens, return_weights=False):  # tokens [N, L] word IDs
        mask, _ = _safe_mask(tokens > 0)
        x = self.proj(self.drop(self.emb(tokens)))
        h, _ = self.mha(x, x, x, key_padding_mask=~mask, need_weights=False)
        v, w = self.att(self.drop(h), mask)
        return (v, w) if return_weights else v


class UserEncoder(nn.Module):
    def __init__(self, d_model, n_heads, query_dim, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.att = AdditiveAttention(d_model, query_dim)
        self.drop = nn.Dropout(dropout)
        self.empty_user = nn.Parameter(torch.zeros(d_model))

    def forward(self, hist_vecs, hist_mask, return_weights=False):   # [B, H, d], [B, H]
        mask, empty = _safe_mask(hist_mask)
        h, _ = self.mha(hist_vecs, hist_vecs, hist_vecs, key_padding_mask=~mask, need_weights=False)
        u, w = self.att(self.drop(h), mask)
        u = torch.where(empty.unsqueeze(1), self.empty_user.to(u.dtype).expand_as(u), u)
        return (u, w) if return_weights else u


class NRMS(nn.Module):
    def __init__(self, word_emb, tokens, cfg):
        super().__init__()
        self.register_buffer("tokens", torch.as_tensor(tokens), persistent=False)
        self.news_encoder = NewsEncoder(word_emb, cfg["d_model"], cfg["n_heads"], cfg["query_dim"], cfg["dropout"])
        self.user_encoder = UserEncoder(cfg["d_model"], cfg["n_heads"], cfg["query_dim"], cfg["dropout"])

    def encode_news(self, rows):
        return self.news_encoder(self.tokens[rows])

    def forward(self, hist_rows, cand_rows):         # [B, H], [B, C] article rows
        B, H = hist_rows.shape
        uniq, inv = torch.unique(torch.cat([hist_rows.reshape(-1), cand_rows.reshape(-1)]), return_inverse=True)
        vecs = self.encode_news(uniq)                  # each distinct article encoded once per batch
        hist_vecs = vecs[inv[:B * H]].view(B, H, -1)
        cand_vecs = vecs[inv[B * H:]].view(B, cand_rows.shape[1], -1)
        user = self.user_encoder(hist_vecs, hist_rows > 0)
        return (cand_vecs * user.unsqueeze(1)).sum(-1)  # raw scores [B, C]

_m = NRMS(WORD_EMB, TOKENS, CFG)
print(f"Parameters: {sum(p.numel() for p in _m.parameters()):,} "
      f"(of which word vectors {_m.news_encoder.emb.weight.numel():,})")
del _m

Parameters: 16,624,248 (of which word vectors 15,917,400)


## 8. Scoring full slates

To score a split, every article is encoded once, each impression's user vector is computed once, and every displayed candidate is scored by a dot product. All candidates are scored (full slates, no sampling). The output matches `utils/metrics.attach_scores`: `impression_id, news_id, score`.

In [9]:
@torch.no_grad()
def encode_all_news(model, batch=2048):
    # [n_articles + 1, d] article vectors; row r belongs to news.loc[r - 1], row 0 is padding
    model.eval()
    out = []
    for s in range(0, model.tokens.shape[0], batch):
        rows = torch.arange(s, min(s + batch, model.tokens.shape[0]), device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            out.append(model.encode_news(rows).float())
    return torch.cat(out)

@torch.no_grad()
def score_candidates(model, cand, H, h_index, batch=1024):
    model.eval()
    news_vecs = encode_all_news(model)
    imp_codes, imp_ids = pd.factorize(cand["impression_id"])
    imp_codes = np.array(imp_codes, dtype=np.int64)
    h_rows = h_index.reindex(imp_ids).to_numpy()
    if np.isnan(h_rows.astype(float)).any():
        raise ValueError("some candidate impressions have no history row")
    h_rows = h_rows.astype(np.int64)
    users = []                                          # one user vector per impression
    for s in range(0, len(imp_ids), batch):
        hist = torch.as_tensor(H[h_rows[s:s + batch]], device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            users.append(model.user_encoder(news_vecs[hist], hist > 0).float())
    users = torch.cat(users)
    news_rows = NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0).to_numpy(dtype=np.int64)
    scores = np.empty(len(cand), dtype=np.float32)
    for s in range(0, len(cand), 200_000):
        u = users[torch.as_tensor(imp_codes[s:s + 200_000], device=DEVICE)]
        v = news_vecs[torch.as_tensor(news_rows[s:s + 200_000], device=DEVICE)]
        scores[s:s + 200_000] = (u * v).sum(-1).cpu().numpy()
    return pd.DataFrame({"impression_id": cand["impression_id"].to_numpy(),
                         "news_id": cand["news_id"].to_numpy(), "score": scores})

def sample_impressions(cand, n, seed):
    ids = cand["impression_id"].drop_duplicates()
    return cand[cand["impression_id"].isin(ids.sample(n=min(n, len(ids)), random_state=seed))]

tune_small = sample_impressions(tune_cand, CFG["n_tune_imps"], CFG["seed"])   # fixed subset for epoch selection
print(f"Tune subset for model selection: {tune_small.impression_id.nunique():,} impressions")

Tune subset for model selection: 20,000 impressions


## 9. Training loop

- **Loss:** softmax cross-entropy over [click, 4 negatives] with target 0, on raw scores.
- **Optimiser:** Adam, learning rate 1e-4, mini-batches of 64.
- **Model selection:** after each epoch the model scores the fixed tune subset (14 Nov, full slates), and the epoch with the best **tune AUC** is kept. Training scores are never used for selection, and the test day is never seen during training or selection.
- **Checkpoints:** saved to Drive after every epoch; `RESUME = True` continues an interrupted run.
- `select_on_tune=False` (used only by the optional refit in 11b) trains for a fixed number of epochs and keeps the last one.

In [10]:
def train_nrms(cfg, run_name, train_H, train_H_rows, train_C, tune=None, select_on_tune=True, log_every=500):
    tune_c, tune_h, tune_i = tune if tune is not None else (tune_small, tune_H, tune_idx)
    seed_everything(cfg["seed"])
    model = NRMS(WORD_EMB, TOKENS, cfg).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    last_ckpt = os.path.join(RUN_DIR, f"{run_name}_last.pt")
    best_ckpt = os.path.join(RUN_DIR, f"{run_name}_best.pt")
    log, start_epoch, best_auc = [], 0, -1.0
    if RESUME and os.path.exists(last_ckpt):
        st = torch.load(last_ckpt, map_location=DEVICE)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        log, start_epoch, best_auc = st["log"], st["epoch"] + 1, st["best_auc"]
        print(f"Resumed {run_name} after epoch {start_epoch}")

    H_dev = torch.as_tensor(train_H, device=DEVICE)
    rows_dev = torch.as_tensor(train_H_rows.astype(np.int64), device=DEVICE)
    C_dev = torch.as_tensor(train_C, device=DEVICE)
    target = torch.zeros(cfg["batch_size"], dtype=torch.long, device=DEVICE)   # the click is column 0
    n, bs = len(C_dev), cfg["batch_size"]
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        t0, total, perm = time.time(), 0.0, torch.randperm(n, device=DEVICE)
        for b, s in enumerate(range(0, n, bs)):
            idx = perm[s:s + bs]
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                logits = model(H_dev[rows_dev[idx]], C_dev[idx])
                loss = F.cross_entropy(logits.float(), target[:len(idx)])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += loss.item() * len(idx)
            if log_every and b % log_every == 0:
                print(f"  epoch {epoch + 1} batch {b:>6}/{math.ceil(n / bs)} loss {loss.item():.4f}")
        row = {"epoch": epoch + 1, "train_loss": total / n}
        if select_on_tune:
            tune_res = evaluate(attach_scores(tune_c, score_candidates(model, tune_c, tune_h, tune_i)), "score")
            row.update({f"tune_{k}": v for k, v in tune_res.items()})
            improved = tune_res["auc"] > best_auc
            best_auc = max(best_auc, tune_res["auc"])
        else:
            improved = True                                       # fixed epochs: keep the latest
        row["minutes"] = (time.time() - t0) / 60
        log.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
        if improved:
            torch.save({"model": model.state_dict(), "cfg": cfg, "epoch": epoch + 1}, best_ckpt)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch,
                    "log": log, "best_auc": best_auc}, last_ckpt)

    best = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(best["model"])
    log = pd.DataFrame(log)
    log.to_csv(os.path.join(RUN_DIR, f"{run_name}_log.csv"), index=False)
    model.best_epoch = int(best["epoch"])
    msg = f"(tune AUC {log.loc[log.epoch == model.best_epoch, 'tune_auc'].item():.4f})" if select_on_tune else "(fixed)"
    print(f"{run_name}: using epoch {model.best_epoch} {msg}")
    return model, log

## 10. Train the main model (fit: 9–13 Nov)

If `RUN` already finished, this cell only loads the best checkpoint.

In [11]:
model, train_log = train_nrms(CFG, RUN, fit_H, train_H_rows, train_C)
with open(os.path.join(RUN_DIR, f"{RUN}_config.json"), "w") as f:
    json.dump({**{k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
               "best_epoch": model.best_epoch, "trained_on": "fit 9-13 Nov"}, f, indent=2)
train_log

Resumed nrms_main_v2 after epoch 6
nrms_main_v2: using epoch 6 (tune AUC 0.6750)


,epoch,train_loss,tune_auc,tune_mrr,tune_ndcg@5,tune_ndcg@10,tune_n_impressions,tune_n_skipped,minutes
0,1,1.349722,0.6515,0.2905,0.3119,0.3739,20000,0,16.386301
1,2,1.286087,0.6570,0.2936,0.3165,0.3787,20000,0,16.396993
2,3,1.263527,0.6678,0.3060,0.3292,0.3920,20000,0,16.440832
3,4,1.250235,0.6725,0.3072,0.3315,0.3930,20000,0,16.484929
4,5,1.239052,0.6749,0.3116,0.3365,0.3984,20000,0,16.461008
5,6,1.229553,0.6750,0.3120,0.3372,0.3985,20000,0,16.424089


## 11. 📊 Results: train vs tune vs test, overall and cold vs warm

All splits are measured the same way: full slates, per-impression metrics from `utils/metrics.py`.
- **Train:** impressions **used in training** (9–13 Nov), re-scored with all their displayed candidates, so train and test are comparable.
- **Tune:** 14 Nov (used to pick the epoch).
- **Test:** 15 Nov (official dev split), scored once with the selected model.

Two extra tables make the results easier to interpret:
- **Random baseline on the same slates:** MRR and nDCG depend on slate length and number of clicks, which differ between segments. Comparing each segment with random scores on its own slates shows the model's real gain.
- **Segment profile:** impressions, average slate length and clicks per impression for cold, warm and mixed.

Score files go to `config.SCORES_DIR` (`data/scores/`).

In [12]:
def full_slates_from_impressions(imp_df, impression_ids, n, seed):
    # Expand 'N1-0 N2-1 ...' slate strings of a sample of the given impressions into candidate rows.
    pool = imp_df[imp_df.impression_id.isin(impression_ids)].drop_duplicates("impression_id")
    sample = pool.sample(n=min(n, len(pool)), random_state=seed)
    rows = sample[["impression_id", "impressions"]].assign(cand=sample["impressions"].str.split()).explode("cand")
    nid_lab = rows["cand"].str.rsplit("-", n=1)
    return pd.DataFrame({"impression_id": rows["impression_id"].to_numpy(),
                         "news_id": nid_lab.str[0].to_numpy(),
                         "label": nid_lab.str[1].astype(int).to_numpy()})

def save_scores(s, name):
    path = os.path.join(SCORE_DIR, f"{name}.parquet")
    s.to_parquet(path, index=False)
    print(f"saved {name}: {len(s):,} rows, columns {list(s.columns)}")

results, scored = {}, {}

# ---- train: training impressions on full slates ----
if COLS["impressions"] in schemas["fit_imp"]:
    slates = pd.read_parquet(os.path.join(PROC, FILES["fit_imp"]), columns=[COLS["impression_id"], COLS["impressions"]])
    slates.columns = ["impression_id", "impressions"]
    tr = full_slates_from_impressions(slates, np.unique(train_imp_ids), CFG["n_train_slate_imps"], CFG["seed"])
    del slates
    tr_H, tr_idx = build_history(fit_imp, impression_ids=tr.impression_id.unique())
    s = score_candidates(model, tr, tr_H, tr_idx)
    save_scores(s, "step4_nrms_train_sample")
    results["train (training impressions)"] = evaluate(attach_scores(tr, s), "score")
else:
    print("fit_impressions has no slate column: train metrics on full slates skipped.")

# ---- tune and test ----
for split, cand, H, idx in [("tune", tune_cand, tune_H, tune_idx), ("eval", eval_cand, eval_H, eval_idx)]:
    s = score_candidates(model, cand, H, idx)
    save_scores(s, f"step4_nrms_{split}")
    scored[split] = attach_scores(cand, s)
    results[split] = evaluate(scored[split], "score")

overall = pd.DataFrame(results).T
overall.to_csv(os.path.join(RUN_DIR, f"{RUN}_overall.csv"))
print("\nOverall (full slates):")
display(overall)

# ---- test by segment, with a random baseline on the same slates ----
test = scored["eval"]
rand = test.assign(score=np.random.default_rng(CFG["seed"]).random(len(test)))
nrms_seg = evaluate_by_segment(test, "score", "is_cold")
rand_seg = evaluate_by_segment(rand, "score", "is_cold")

seg = label_impression_segment(test, "is_cold")
slate = test.groupby("impression_id").agg(slate_len=("news_id", "size"), clicks=("label", "sum")).join(seg)
profile = slate.groupby("segment")[["slate_len", "clicks"]].mean().round(1)
profile.loc["all"] = slate[["slate_len", "clicks"]].mean().round(1)

by_segment = pd.DataFrame({
    "impressions": nrms_seg["n_impressions"],
    "avg_slate_len": profile["slate_len"], "avg_clicks": profile["clicks"],
    "AUC": nrms_seg["auc"], "AUC_random": rand_seg["auc"],
    "MRR": nrms_seg["mrr"], "MRR_random": rand_seg["mrr"],
    "nDCG@10": nrms_seg["ndcg@10"], "nDCG@10_random": rand_seg["ndcg@10"],
}).reindex(["all", "cold", "warm", "mixed"]).dropna(how="all")
by_segment["MRR_lift"] = (by_segment["MRR"] - by_segment["MRR_random"]).round(4)
by_segment.to_csv(os.path.join(RUN_DIR, f"{RUN}_eval_by_segment.csv"))
print("\nTest (15 Nov) by impression segment (cold = every clicked article was cold):")
display(by_segment)

# ---- how much of each day is new content? (explains the tune -> test drop) ----
freshness = pd.DataFrame({
    "cold share of candidates": [tune_cand.is_cold.mean(), eval_cand.is_cold.mean()],
    "cold share of clicks": [tune_cand.loc[tune_cand.label == 1, "is_cold"].mean(),
                             eval_cand.loc[eval_cand.label == 1, "is_cold"].mean()],
    "AUC": [results["tune"]["auc"], results["eval"]["auc"]],
}, index=["tune (14 Nov)", "test (15 Nov)"]).round(4)
display(freshness)

saved step4_nrms_train_sample: 799,703 rows, columns ['impression_id', 'news_id', 'score']
saved step4_nrms_tune: 17,400,106 rows, columns ['impression_id', 'news_id', 'score']
saved step4_nrms_eval: 14,085,557 rows, columns ['impression_id', 'news_id', 'score']

Overall (full slates):


,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
train (training impressions),0.7683,0.3992,0.4506,0.5069,20000.0,0.0
tune,0.6757,0.3109,0.3365,0.3984,431517.0,0.0
eval,0.6288,0.2907,0.3162,0.3829,376471.0,0.0



Test (15 Nov) by impression segment (cold = every clicked article was cold):


,impressions,avg_slate_len,avg_clicks,AUC,AUC_random,MRR,MRR_random,nDCG@10,nDCG@10_random,MRR_lift
all,376471,37.4,1.5,0.6288,0.5008,0.2907,0.2189,0.3829,0.2866,0.0718
cold,77199,24.5,1.1,0.5767,0.5006,0.3500,0.2986,0.4426,0.3819,0.0514
warm,262807,37.7,1.4,0.6411,0.5010,0.2897,0.2102,0.3777,0.2734,0.0795
mixed,36465,63.1,3.3,0.6503,0.5002,0.1729,0.1128,0.2938,0.1796,0.0601


,cold share of candidates,cold share of clicks,AUC
tune (14 Nov),0.2099,0.2543,0.6757
test (15 Nov),0.1793,0.2346,0.6288


### Interpretation (30 Oct run: 1.5M clicks, 6 epochs; update if rerun)

- **Overall:** train 0.766 → tune 0.676 → test **0.629** AUC. Train and test are measured on equal terms (full slates), so the gap is a real generalisation gap, not a candidate-set artefact.
- **Tune → test drop:** more training lifted tune AUC by 0.027 but test only by 0.003. Tune (14 Nov) is the day right after training, while by 15 Nov much of the article pool has turned over (median article lifespan is about 13 hours, interim Section 3.4). The model learns article-specific patterns that age within a day. The freshness table above shows how the share of new content differs between the two days.
- **Cold vs warm:** cold-click impressions score AUC **0.577** vs **0.641** for warm. Even a content-based model ranks new articles worse, which is the gap our business objective targets. Compare segments with **AUC**, or with the lift over random, not raw MRR/nDCG: cold impressions have different slate lengths and click counts, which change MRR/nDCG even for random scores.

## 11b. Optional: final refit on 9–14 Nov

Standard practice after model selection: retrain once on fit + tune (9–14 Nov) with the selected settings and number of epochs, then score the test day. This gives the model the freshest articles before test day, without using the test day. Tune can no longer select epochs (it is now training data), so the refit trains for exactly the selected number of epochs.

Set `REFIT_FINAL = True` in section 1 to run it. Its scores are saved separately (`step4_nrms_refit_eval.parquet`), so the main results stay intact.

In [13]:
FINAL_MODEL, FINAL_NAME, FINAL_TRAINED_ON = model, RUN, "fit 9-13 Nov"
if REFIT_FINAL:
    refit_cfg = dict(CFG, epochs=model.best_epoch)
    tune_ids, tune_C = build_train_samples(tune_cand, None, CFG["neg_k"], CFG["seed"])
    all_H = np.concatenate([fit_H, tune_H])                            # tune rows come after fit rows
    all_rows = np.concatenate([train_H_rows.astype(np.int64),
                               tune_idx.reindex(tune_ids).to_numpy().astype(np.int64) + len(fit_H)])
    all_C = np.concatenate([train_C, tune_C])
    print(f"Refit samples: {len(all_C):,} ({len(train_C):,} fit + {len(tune_C):,} tune), epochs {refit_cfg['epochs']}")
    refit_model, refit_log = train_nrms(refit_cfg, RUN + "_refit", all_H, all_rows, all_C, select_on_tune=False)
    s = score_candidates(refit_model, eval_cand, eval_H, eval_idx)
    save_scores(s, "step4_nrms_refit_eval")
    refit_scored = attach_scores(eval_cand, s)
    refit_table = pd.DataFrame({"main (9-13 Nov)": results["eval"], "refit (9-14 Nov)": evaluate(refit_scored, "score")}).T
    display(refit_table)
    display(evaluate_by_segment(refit_scored, "score", "is_cold"))
    FINAL_MODEL, FINAL_NAME, FINAL_TRAINED_ON = refit_model, RUN + "_refit", "fit + tune 9-14 Nov"
print("Final model for export:", FINAL_NAME, "| trained on", FINAL_TRAINED_ON)

Final model for export: nrms_main_v2 | trained on fit 9-13 Nov


## 11c. Export article vectors (shared)

The final model's 256-d article vectors (the news encoder output, i.e. what the model compares with the user vector) are saved for the rest of the team:
- `data/embeddings/step4_nrms_news_vectors.npy`: float32 array `[n_articles, 256]`
- `data/embeddings/step4_nrms_news_ids.parquet`: `news_id` in the same row order
- `data/embeddings/step4_nrms_news_vectors.json`: what the vectors are and how they were made

The vectors are **not normalised**. For cosine similarity, divide each row by its length first.

In [14]:
vecs = encode_all_news(FINAL_MODEL).cpu().numpy()[1:].astype(np.float32)   # drop padding row 0
ids = news[["news_id"]].copy()
assert len(vecs) == len(ids) and ids.news_id.is_unique

np.save(os.path.join(EMB_DIR, "step4_nrms_news_vectors.npy"), vecs)
ids.to_parquet(os.path.join(EMB_DIR, "step4_nrms_news_ids.parquet"), index=False)
with open(os.path.join(EMB_DIR, "step4_nrms_news_vectors.json"), "w") as f:
    json.dump({"model": FINAL_NAME, "epoch": FINAL_MODEL.best_epoch, "trained_on": FINAL_TRAINED_ON,
               "dim": int(vecs.shape[1]), "n_articles": int(len(vecs)), "normalised": False,
               "source": "NRMS news encoder output (title words, word2vec-initialised, fine-tuned)",
               "row_order": "same as step4_nrms_news_ids.parquet"}, f, indent=2)

norms = np.linalg.norm(vecs, axis=1)
print(f"Saved {vecs.shape[0]:,} article vectors of size {vecs.shape[1]} to {EMB_DIR}")
print(f"Vector length: median {np.median(norms):.2f}, min {norms.min():.2f}, max {norms.max():.2f}")

Saved 104,151 article vectors of size 256 to /content/drive/MyDrive/BT4222_Project/data/embeddings
Vector length: median 1.85, min 0.60, max 6.76


## 12. Ablations (architecture, aspect 3)

Each variant changes one setting relative to a base model. To fit the time budget, ablations train on 200k clicks for 2 epochs and are compared on the **tune split only**; the test day is used once, in section 11. Finished ablations reload from their checkpoints.

In [15]:
ABL_BASE = dict(CFG, n_train_clicks=200_000, epochs=2)
ABLATIONS = {
    "base (title, 16 heads, hist 50)": {},
    "history 10":                      {"max_hist": 10},
    "4 heads":                         {"n_heads": 4},
    "title + abstract":                {"text_fields": ("title", "abstract"), "field_lens": (30, 50)},
}

def run_ablation(name, overrides):
    global TOKENS, CFG
    cfg = dict(ABL_BASE, **overrides)
    saved = (TOKENS, CFG)
    try:
        CFG = cfg                         # build_history and the model read the history length from CFG
        if cfg["text_fields"] != saved[1]["text_fields"]:
            TOKENS = build_tokens(cfg["text_fields"], cfg["field_lens"], VOCAB)
        imp_ids, C = build_train_samples(fit_cand, cfg["n_train_clicks"], cfg["neg_k"], cfg["seed"])
        H, idx = build_history(fit_imp, impression_ids=np.unique(imp_ids))
        t_H, t_idx = build_history(tune_imp)
        run = "abl_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        m, log = train_nrms(cfg, run, H, idx.reindex(imp_ids).to_numpy(), C,
                            tune=(tune_small, t_H, t_idx), log_every=0)
        res = evaluate(attach_scores(tune_small, score_candidates(m, tune_small, t_H, t_idx)), "score")
        return {"best_epoch": m.best_epoch, **res}
    finally:
        TOKENS, CFG = saved

if RUN_ABLATIONS:
    ablation_table = pd.DataFrame({name: run_ablation(name, o) for name, o in ABLATIONS.items()}).T
    ablation_table["AUC_vs_base"] = (ablation_table["auc"] - ablation_table["auc"].iloc[0]).round(4)
    ablation_table.to_csv(os.path.join(RUN_DIR, "ablation_tune.csv"))
    display(ablation_table)

Resumed abl_base_title_16_heads_hist_50 after epoch 2
abl_base_title_16_heads_hist_50: using epoch 2 (tune AUC 0.6238)
Resumed abl_history_10 after epoch 2
abl_history_10: using epoch 2 (tune AUC 0.6202)
Resumed abl_4_heads after epoch 2
abl_4_heads: using epoch 2 (tune AUC 0.6286)
Resumed abl_title_abstract after epoch 2
abl_title_abstract: using epoch 2 (tune AUC 0.6221)


,best_epoch,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped,AUC_vs_base
"base (title, 16 heads, hist 50)",2.0,0.6238,0.2646,0.2806,0.3445,20000.0,0.0,0.0000
history 10,2.0,0.6202,0.2671,0.2839,0.3474,20000.0,0.0,-0.0036
4 heads,2.0,0.6286,0.2706,0.2866,0.3516,20000.0,0.0,0.0048
title + abstract,2.0,0.6221,0.2658,0.2810,0.3461,20000.0,0.0,-0.0017


**Interpretation (30 Oct run):** all variants land within 0.620–0.629 tune AUC (history 50 vs 10: +0.004; 4 heads slightly best; abstract: no gain). The main model (1.5M clicks, 6 epochs) reaches 0.676 on tune, so at this scale **training data matters far more than these architecture choices**.

## 13. Inside the model: attention weights (aspect 4)

For a few test impressions:
- **Word attention:** which title words the news encoder weighted most for the clicked article.
- **History attention:** which past clicks the user encoder weighted most.

To read a weight, compare it with equal attention: 1 ÷ the number of items (about 0.12 for an 8-word title, 0.02 for 50 history articles).

Caveat (cf. KGIN, Topic 9): attention weights show what the model relied on to predict, not verified reasons why the user clicked.

In [16]:
# Free the large training tables: from here on only the model, news and the test split are needed.
for _name in ["fit_cand", "fit_imp", "fit_H", "train_C", "train_H_rows", "train_imp_ids", "tr", "tr_H", "data",
              "all_H", "all_rows", "all_C"]:
    globals().pop(_name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

INV_VOCAB = {i: w for w, i in VOCAB.items()}
NEWS_VECS = encode_all_news(model)          # every article encoded once, reused below

@torch.no_grad()
def explain(model, impression_id, top=5):
    model.eval()
    hist = torch.as_tensor(eval_H[[eval_idx[impression_id]]], device=DEVICE)
    _, hw = model.user_encoder(NEWS_VECS[hist], hist > 0, return_weights=True)
    hrows, hw = hist[0].cpu().numpy(), hw[0].float().cpu().numpy()
    n_hist = int((hrows > 0).sum())
    print(f"Impression {impression_id}: most-attended history articles (equal attention = {1 / max(n_hist, 1):.2f})")
    for j in np.argsort(-hw)[:3]:
        if hrows[j] > 0:
            print(f"   {hw[j]:.2f}  {news.loc[hrows[j] - 1, 'title']}")
    clicked = eval_cand.loc[(eval_cand.impression_id == impression_id) & (eval_cand.label == 1), "news_id"]
    for nid in clicked:
        row = int(NEWS_ROW[nid])
        tok = model.tokens[[row]]
        _, ww = model.news_encoder(tok, return_weights=True)
        words = [(INV_VOCAB[int(t)], float(w)) for t, w in zip(tok[0].cpu(), ww[0].float().cpu()) if int(t) > 0]
        print(f"Clicked: {news.loc[row - 1, 'title']}")
        print("   top words:", ", ".join(f"{w} ({a:.2f})" for w, a in sorted(words, key=lambda x: -x[1])[:top]))

examples = eval_cand.loc[eval_cand.label == 1, "impression_id"].drop_duplicates().sample(3, random_state=CFG["seed"])
for imp in examples:
    explain(model, imp)
    print()

Impression 90453: most-attended history articles (equal attention = 0.07)
   0.08  'Go to confession': Scaramucci urges Mick Mulvaney to leave the White House
   0.07  Democrats' 2020 race has a new shadow: Hillary Clinton
   0.07  Joe Biden reportedly denied Communion at a South Carolina church because of his stance on abortion
Clicked: The Unlikely Star of My Family's Thanksgiving Table
   top words: my (0.14), family's (0.13), table (0.13), of (0.13), the (0.12)

Impression 210745: most-attended history articles (equal attention = 0.02)
   0.03  Sessions wins endorsement from key GOP senators
   0.03  Loved One Gave You Down Payment Money? You'll Need a Mortgage Gift Letter
   0.03  Exclusive: Ukraine to fire prosecutor who discussed Bidens with Giuliani - source
Clicked: This Easy Trick Will Defrost Your Car Window In Seconds
   top words: defrost (0.12), trick (0.10), seconds (0.10), your (0.10), easy (0.10)

Impression 192076: most-attended history articles (equal attention = 0.0

### 13b. Attention in numbers

Measured over thousands of test impressions:
1. **History attention vs interests:** how often the *most-attended* history article has the same category as the clicked article, compared with a *random* history article.
2. **Word attention:** average weight on stopwords (*the, my, of…*) vs content words, relative to equal attention (1.0 = every word weighted the same).

In [17]:
STOPWORDS = set("the a an of to in on for and or with at by from is are was were be been this that these those it its "
                "he she they his her their my your our you we i as after says how why what when who new".split())
CATEGORY = news["category"].to_numpy()          # article row r -> CATEGORY[r - 1]

@torch.no_grad()
def attention_stats(model, n_imps=3000, seed=CFG["seed"]):
    model.eval()
    rng = np.random.default_rng(seed)
    clicks = eval_cand.loc[eval_cand.label == 1, ["impression_id", "news_id"]]
    imps = clicks["impression_id"].drop_duplicates()
    imps = imps.sample(n=min(n_imps, len(imps)), random_state=seed).to_numpy()
    clicked_cats = (clicks.assign(cat=CATEGORY[NEWS_ROW.reindex(clicks.news_id).to_numpy() - 1])
                          .groupby("impression_id")["cat"].agg(set))

    # 1. history attention: top-attended vs random history article
    top_hit, rand_hit = [], []
    for s in range(0, len(imps), 1000):
        batch = imps[s:s + 1000]
        hist = torch.as_tensor(eval_H[eval_idx.reindex(batch).to_numpy()], device=DEVICE)
        _, hw = model.user_encoder(NEWS_VECS[hist], hist > 0, return_weights=True)
        hist, hw = hist.cpu().numpy(), hw.float().cpu().numpy()
        for imp, h, w in zip(batch, hist, hw):
            real = np.flatnonzero(h > 0)
            if len(real) < 2:
                continue
            cats = clicked_cats[imp]
            top_hit.append(CATEGORY[h[real[np.argmax(w[real])]] - 1] in cats)
            rand_hit.append(CATEGORY[h[rng.choice(real)] - 1] in cats)

    # 2. word attention: stopwords vs content words (weight x number of words; 1.0 = equal)
    rows = NEWS_ROW.reindex(clicks.loc[clicks.impression_id.isin(imps), "news_id"].unique()).to_numpy()
    stop_w, content_w = [], []
    for s in range(0, len(rows), 2000):
        tok = model.tokens[torch.as_tensor(rows[s:s + 2000], device=DEVICE)]
        _, ww = model.news_encoder(tok, return_weights=True)
        tok, ww = tok.cpu().numpy(), ww.float().cpu().numpy()
        for t, w in zip(tok, ww):
            real = t > 0
            n = real.sum()
            for ti, wi in zip(t[real], w[real]):
                (stop_w if INV_VOCAB[int(ti)] in STOPWORDS else content_w).append(wi * n)

    return pd.DataFrame({
        "value": [np.mean(top_hit), np.mean(rand_hit), np.mean(stop_w), np.mean(content_w)],
        "n": [len(top_hit), len(rand_hit), len(stop_w), len(content_w)],
    }, index=["history: top-attended article shares clicked category",
              "history: random article shares clicked category",
              "words: relative weight on stopwords (1 = equal)",
              "words: relative weight on content words (1 = equal)"]).round(3)

attention_table = attention_stats(model)
attention_table.to_csv(os.path.join(RUN_DIR, f"{RUN}_attention_stats.csv"))
display(attention_table)

,value,n
history: top-attended article shares clicked category,0.283,2861
history: random article shares clicked category,0.257,2861
words: relative weight on stopwords (1 = equal),0.941,1669
words: relative weight on content words (1 = equal),1.018,5394


**Interpretation (30 Oct run):** the most-attended history article shares the clicked category 28.3% of the time vs 25.7% for a random history article, so the user encoder leans towards interest-relevant clicks. The effect is modest because NRMS builds one user vector for all candidates (candidate-independent attention); candidate-aware attention is a natural extension. Word attention down-weights stopwords (0.94 vs 1.02 for content words) but stays close to uniform: title meaning is carried mostly by the word vectors and self-attention.

## 14. Summary and hand-off

| | Value (30 Oct run; update if rerun) |
|---|---|
| Training data | 1.5M clicks from fit (9–13 Nov), 1 click + 4 same-impression negatives |
| Selected epoch | best tune AUC (see section 10) |
| Test (15 Nov), full slates | AUC 0.629, MRR 0.291, nDCG@5 0.316, nDCG@10 0.383 |
| Cold vs warm (test AUC) | 0.577 vs 0.641 |
| Train vs tune vs test AUC | 0.766 / 0.676 / 0.629 |
| Ablations | within 0.620–0.629 tune AUC; training data matters more |

**Files for the team**
- `data/scores/step4_nrms_eval.parquet`: test-day scores, 14,085,557 rows, `impression_id, news_id, score` (raw dot product, higher = more relevant, can be negative). Rescale within each impression (e.g. min-max) before combining with similarities in MMR.
- `data/scores/step4_nrms_tune.parquet`, `step4_nrms_train_sample.parquet`: tune-day and training-impression scores.
- `data/embeddings/step4_nrms_news_vectors.npy` + `step4_nrms_news_ids.parquet`: NRMS article vectors. For a fair diversity measure, MMR should optimise with one embedding and diversity should be evaluated with a different, independent one (e.g. step 3's text embeddings or TF-IDF), plus category counts.